In [27]:
pip install -U pydantic

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 12.7 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.27.2
    Not uninstalling pydantic-core at /opt/conda/lib/python3.11/site-packages, outside environment /root/.clearml/venvs-builds/3.11
    Can't uninstall 'pydantic_core'. No files were found to uninstall.
  Attempting uninstall: pydantic
    Found existing installation: pydantic 1.10.22
    Uninstalling pydantic-1.10.22:
      Successfully uninstalled pydantic-1.10.22
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastapi 0.89.1 requires pydantic!=1.7,!=1.7.1,!=1.7.2,!=1.7.3,!=1.8,!=1.8.1,<2.0.0,>=1.6.2, but you have pydantic 2.11.7 which is incompatible.
mistral-common 1.5.3 requires numpy>=1.25; python_version >= "3.9", but you have numpy 1.23.5 which is incompatible.
vllm 0.6.6 requires fastap

In [1]:
import json
import torch
import numpy as np

In [2]:
from deeppavlov import build_model, train_model, train_evaluate_model_from_config, evaluate_model
from deeppavlov.core.common.file import read_json

In [3]:
from deeppavlov.dataset_readers.hallucination_detection_reader import HallucinationDatasetReader, RAGTruthDatasetReader

In [4]:
from transformers import AutoModel, AutoConfig, AutoTokenizer, AutoModelForTokenClassification

In [5]:
from deeppavlov.core.data.data_learning_iterator import DataLearningIterator
from deeppavlov.models.preprocessors.torch_transformers_preprocessor import TorchTransformersHallucinationDetectorPreprocessor 
from deeppavlov.metrics.fmeasure import token_binary_f1, token_binary_precision, token_binary_recall

In [6]:
device = 'cuda'

In [7]:
path = 'deeppavlov/configs/hallucination_detection/ragtruth_modernbert_base.json'
# path = 'deeppavlov/configs/hallucination_detection/ragtruth_modernbert_large.json'
# path = 'deeppavlov/configs/hallucination_detection/ragtruth_mbert_base.json'
# path = 'deeppavlov/configs/hallucination_detection/ragtruth_deberta_small.json'


# path = 'deeppavlov/configs/hallucination_detection/ragtruth_eurobert_base.json'

In [8]:
config = read_json(path)

In [9]:
model = build_model(config, load_trained=True)

Some weights of ModernBertForTokenClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
# model_name = "KRLabsOrg/lettucedect-base-modernbert-en-v1"
model_name = "KRLabsOrg/lettucedect-large-modernbert-en-v1"

main_model = AutoModelForTokenClassification.from_pretrained(model_name)
main_model.to(device)
main_model.eval()
print()

model.safetensors:   3%|3         | 52.4M/1.64G [00:00<?, ?B/s]

In [11]:
# evaluate_model(config)

In [12]:
ds = RAGTruthDatasetReader().read("/root/.deeppavlov/downloads/ragtruth/", validation_split=0.10)

2025-06-30 11:14:55.583 INFO in 'deeppavlov.dataset_readers.hallucination_detection_reader'['hallucination_detection_reader'] at line 52: Preprocessing raw RAGTruth data...
2025-06-30 11:14:55.585 INFO in 'deeppavlov.dataset_readers.hallucination_detection_reader'['hallucination_detection_reader'] at line 148: Loading raw data from /root/.deeppavlov/downloads/ragtruth/response.jsonl and /root/.deeppavlov/downloads/ragtruth/source_info.jsonl
2025-06-30 11:14:55.789 INFO in 'deeppavlov.dataset_readers.hallucination_detection_reader'['hallucination_detection_reader'] at line 200: Loaded 17790 responses and 2965 sources
2025-06-30 11:14:55.979 INFO in 'deeppavlov.dataset_readers.hallucination_detection_reader'['hallucination_detection_reader'] at line 174: Preprocessed 17790 samples from raw data
2025-06-30 11:14:56.726 INFO in 'deeppavlov.dataset_readers.hallucination_detection_reader'['hallucination_detection_reader'] at line 240: Saved preprocessed data to /root/.deeppavlov/downloads/ra

In [14]:
len(ds['train'])

13581

In [9]:
iterator = DataLearningIterator(ds)
preprocessor = TorchTransformersHallucinationDetectorPreprocessor(
    config['metadata']['variables']['TRANSFORMER'], 
    do_lower_case=config['chainer']['pipe'][0]['do_lower_case'],
    max_seq_length=config['chainer']['pipe'][0]['max_seq_length']
)

NameError: name 'ds' is not defined

In [10]:
tokenizer = AutoTokenizer.from_pretrained(config['metadata']['variables']['TRANSFORMER'])
do_lower_case = config['chainer']['pipe'][0]['do_lower_case']
max_seq_length = config['chainer']['pipe'][0]['max_seq_length']

In [15]:
from tqdm import tqdm

gt_spans = []
pred_spans = []
task_types = []

for batch in tqdm(iterator.gen_batches(1, data_type='test'), total=len(ds['test'])):
    x, y = batch
    prompt = x[0]['prompt']
    answer = x[0]['answer']
    task_types.append(x[0]['task_type'])
    gt_spans.append(x[0]['labels'])

    encoding, _, offsets, answer_start_token = TorchTransformersHallucinationDetectorPreprocessor.prepare_tokenized_input(
            tokenizer, prompt, answer, max_seq_length
        )
    labels = torch.full_like(encoding.input_ids[0], -100, device=device)
    labels[answer_start_token:] = 0
    encoding = {
            key: value.to(device)
            for key, value in encoding.items()
            if key in ["input_ids", "attention_mask", "labels"]
        }
    with torch.inference_mode():
        outputs = main_model(**encoding)
    logits = outputs.logits
    token_preds = torch.argmax(logits, dim=-1)[0]
    probabilities = torch.softmax(logits, dim=-1)[0]
    token_preds = torch.where(labels == -100, labels, token_preds)

    if answer_start_token < offsets.size(0):
        answer_char_offset = offsets[answer_start_token][0].item()
    else:
        answer_char_offset = 0

    spans: list[dict] = []
    current_span: dict | None = None

    for i in range(answer_start_token, token_preds.size(0)):
        # Skip tokens marked as ignored.
        if labels[i].item() == -100:
            continue

        token_start, token_end = offsets[i].tolist()
        # Skip special tokens with zero length.
        if token_start == token_end:
            continue

        # Adjust offsets relative to the answer text.
        rel_start = token_start - answer_char_offset
        rel_end = token_end - answer_char_offset

        is_hallucination = (
            token_preds[i].item() == 1
        )  # assuming class 1 indicates hallucination.
        confidence = probabilities[i, 1].item() if is_hallucination else 0.0

        if is_hallucination:
            if current_span is None:
                current_span = {
                    "start": rel_start,
                    "end": rel_end,
                    "confidence": confidence,
                }
            else:
                # Extend the current span.
                current_span["end"] = rel_end
                current_span["confidence"] = max(current_span["confidence"], confidence)
        else:
            # If we were building a hallucination span, finalize it.
            if current_span is not None:
                # Extract the hallucinated text from the answer.
                span_text = answer[current_span["start"] : current_span["end"]]
                current_span["text"] = span_text
                spans.append(current_span)
                current_span = None

    # Append any span still in progress.
    if current_span is not None:
        span_text = answer[current_span["start"] : current_span["end"]]
        current_span["text"] = span_text
        spans.append(current_span)
    pred_spans.append(spans)


100%|██████████| 2700/2700 [03:57<00:00, 11.38it/s]


In [21]:
import pandas as pd

In [22]:
df = pd.DataFrame({
    "task_type": task_types,
    "pred_spans": pred_spans,
    "gt_spans": gt_spans,
})

In [23]:
# df.to_csv('ragtruth_modern_mbert_base_large.csv',index=False)

In [24]:
def calc_span_level_metrics(gt_spans, pred_spans):
    total_overlap = 0
    total_predicted = 0
    total_gold = 0
    for gold_spans, predicted_spans in zip(gt_spans, pred_spans):
        sample_predicted_length = sum(pred["end"] - pred["start"] for pred in predicted_spans)
        total_predicted += sample_predicted_length
    
        # Compute total gold span length once for this sample.
        sample_gold_length = sum(gold["end"] - gold["start"] for gold in gold_spans)
        total_gold += sample_gold_length
    
        # Now, compute the overlap between each predicted span and each gold span.
        sample_overlap = 0
        for pred in predicted_spans:
            for gold in gold_spans:
                overlap_start = max(pred["start"], gold["start"])
                overlap_end = min(pred["end"], gold["end"])
                if overlap_end > overlap_start:
                    sample_overlap += overlap_end - overlap_start
        total_overlap += sample_overlap
    
    precision = total_overlap / total_predicted if total_predicted > 0 else 0
    recall = total_overlap / total_gold if total_gold > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    results = {"precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4)}
    return results

In [25]:
for task_type, subdf in df.groupby('task_type'):
    print(task_type)
    print(calc_span_level_metrics(subdf.pred_spans.values, subdf.gt_spans.values))
print('overall')
print(calc_span_level_metrics(df.pred_spans.values, df.gt_spans.values))


Data2txt
{'precision': 0.5599, 'recall': 0.6471, 'f1': 0.6004}
QA
{'precision': 0.6214, 'recall': 0.6685, 'f1': 0.6441}
Summary
{'precision': 0.3547, 'recall': 0.6017, 'f1': 0.4463}
overall
{'precision': 0.5396, 'recall': 0.6492, 'f1': 0.5893}


In [ ]:
# 'KRLabsOrg/lettucedect-large-modernbert-en-v1'

# Data2txt
# {'precision': 0.5599, 'recall': 0.6471, 'f1': 0.6004}
# QA
# {'precision': 0.6214, 'recall': 0.6685, 'f1': 0.6441}
# Summary
# {'precision': 0.3547, 'recall': 0.6017, 'f1': 0.4463}
# overall
# {'precision': 0.5396, 'recall': 0.6492, 'f1': 0.5893}

In [25]:
# 'KRLabsOrg/lettucedect-base-modernbert-en-v1'
# Data2txt
# {'precision': 0.5657, 'recall': 0.5824, 'f1': 0.5739}
# QA
# {'precision': 0.604, 'recall': 0.6263, 'f1': 0.615}
# Summary
# {'precision': 0.2808, 'recall': 0.5298, 'f1': 0.3671}
# overall
# {'precision': 0.5201, 'recall': 0.5935, 'f1': 0.5544}

In [21]:
# Deberta BASE
# Data2txt
# {'precision': 0.568, 'recall': 0.6201, 'f1': 0.5929}
# QA
# {'precision': 0.6262, 'recall': 0.6152, 'f1': 0.6207}
# Summary
# {'precision': 0.3024, 'recall': 0.6173, 'f1': 0.406}
# overall
# {'precision': 0.5338, 'recall': 0.6177, 'f1': 0.5727}

In [18]:
# Modern Large
# Data2txt
# {'precision': 0.5302, 'recall': 0.6478, 'f1': 0.5831}
# QA
# {'precision': 0.6444, 'recall': 0.7358, 'f1': 0.6871}
# Summary
# {'precision': 0.3058, 'recall': 0.726, 'f1': 0.4304}
# overall
# {'precision': 0.5252, 'recall': 0.6945, 'f1': 0.5981}

In [ ]:
# Modern Base
# Data2txt
# {'precision': 0.5078, 'recall': 0.6479, 'f1': 0.5694}
# QA
# {'precision': 0.5752, 'recall': 0.7105, 'f1': 0.6357}
# Summary
# {'precision': 0.2665, 'recall': 0.6347, 'f1': 0.3754}
# overall
# {'precision': 0.4821, 'recall': 0.6723, 'f1': 0.5615}

In [62]:
# EuroBERT
# Data2txt
# {'precision': 0.5647749834546658, 'recall': 0.5762964630406033, 'f1': 0.5704775566480509}
# QA
# {'precision': 0.6283600025298842, 'recall': 0.6516035941496688, 'f1': 0.6397707514971989}
# Summary
# {'precision': 0.21977655494413872, 'recall': 0.41594782242794026, 'f1': 0.28759501036476703}
# overall
# {'precision': 0.515912293163478, 'recall': 0.5865182223751969, 'f1': 0.548954254844625}